# Inverse-RL Colab Notebook

Single resumable notebook for the inverse-RL experiment.

In [1]:
# Cell 0 — setup (repo, branch, dependencies, GPU)

import os

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
os.environ["VLLM_USE_V1"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import subprocess
from pathlib import Path
from importlib import metadata

# =========================
# CONFIG
# =========================

GIT_URL = os.environ.get(
    "INVERSE_RL_GIT_URL",
    "https://github.com/shengweiming/inverse-RL.git",
)

BRANCH = "codex/implement-stage-1-forward-rft-and-inverse-baseline"  # <-- change branch here when needed

REPO_DIR = Path("/content/inverse-RL")

# vLLM version known to work on Colab CUDA 12.8
INSTALL_VLLM = True
VLLM_VERSION = "0.10.2"
FORCE_REINSTALL_VLLM = False  # set True if vLLM breaks again

def run(cmd):
    print("[run]", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True)

# =========================
# CLONE REPO IF NEEDED
# =========================

if REPO_DIR.exists():
    print(f"[skip] repo already exists at {REPO_DIR}")
else:
    print(f"[run] cloning {GIT_URL} -> {REPO_DIR}")
    run(["git", "clone", GIT_URL, str(REPO_DIR)])

# =========================
# MOVE INTO REPO
# =========================

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# =========================
# FETCH + CHECKOUT BRANCH
# =========================

print("[run] fetching latest branches")
run(["git", "fetch", "--all"])

print(f"[run] checking out branch: {BRANCH}")
run(["git", "checkout", BRANCH])

print("[run] pulling latest changes")
run(["git", "pull"])

# =========================
# SHOW REPO INFO
# =========================

print("[run] current branch:")
run(["git", "branch", "--show-current"])

print("[run] current commit:")
run(["git", "rev-parse", "HEAD"])

print("[run] repo files:")
run(["ls"])

# =========================
# INSTALL REQUIREMENTS
# =========================

requirements_path = REPO_DIR / "requirements.txt"

if requirements_path.exists():
    print("[run] installing requirements")
    run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirements_path),
    ])
else:
    print("[skip] no requirements.txt found")

# =========================
# PIN PEFT (avoid torchao>=0.16 import gate in newer peft)
# =========================
print("[run] pinning peft")
run([
    sys.executable, "-m", "pip", "install", "-q", "peft==0.13.2",
])

# =========================
# INSTALL vLLM COMPATIBLY FOR COLAB
# =========================

if INSTALL_VLLM:
    try:
        installed_vllm = metadata.version("vllm")
    except metadata.PackageNotFoundError:
        installed_vllm = None

    needs_vllm_install = (
        FORCE_REINSTALL_VLLM
        or installed_vllm != VLLM_VERSION
    )

    if needs_vllm_install:
        print(f"[run] installing vLLM=={VLLM_VERSION} with CUDA 12.8 backend")

        run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm", "vllm-flash-attn"])
        run([sys.executable, "-m", "pip", "cache", "purge"])
        run([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"])
        run(["uv", "pip", "install", "--system", f"vllm=={VLLM_VERSION}", "--torch-backend=cu128"])

        print("[note] If this is the first vLLM install in this runtime, restart runtime before importing vLLM.")
    else:
        print(f"[skip] vLLM=={VLLM_VERSION} already installed")

# =========================
# PIN TOKENIZER STACK FOR vLLM 0.10.2
# =========================
print("[run] pinning transformers/tokenizers for vLLM compatibility")
run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--force-reinstall",
    "transformers==4.56.2",
    "tokenizers==0.22.1",
    "huggingface-hub>=0.34.0",
])

# =========================
# GPU INFO
# =========================

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

[skip] repo already exists at /content/inverse-RL
[run] fetching latest branches
[run] git fetch --all
[run] checking out branch: codex/implement-stage-1-forward-rft-and-inverse-baseline
[run] git checkout codex/implement-stage-1-forward-rft-and-inverse-baseline
[run] pulling latest changes
[run] git pull
[run] current branch:
[run] git branch --show-current
[run] current commit:
[run] git rev-parse HEAD
[run] repo files:
[run] ls
[run] installing requirements
[run] /usr/bin/python3 -m pip install -q -r /content/inverse-RL/requirements.txt
[skip] vLLM==0.10.2 already installed
[run] pinning transformers/tokenizers for vLLM compatibility
[run] /usr/bin/python3 -m pip install -q --force-reinstall transformers==4.56.2 tokenizers==0.22.1 huggingface-hub>=0.34.0
NVIDIA A100-SXM4-80GB, 81920 MiB


In [2]:
# Cell 1 — Drive + config + resumable state
import json
import os
from pathlib import Path

from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/inverse-rl")
DATA_DIR = ROOT / "data"
CKPT_DIR = ROOT / "ckpts"
RESULTS_DIR = ROOT / "results"
for d in (ROOT, DATA_DIR, CKPT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

CFG = {
    "model_name": "meta-llama/Llama-3.2-3B-Instruct",
    "shakedown_model": "meta-llama/Llama-3.2-1B-Instruct",
    "lora_r": 32,
    "G": 8,
    "max_prompt_len": 2048,
    "max_completion_len": 512,
    "lr": 2e-6,
    "train_levels": [1],
    "composition_train_levels": [2],
    "eval_levels": [1, 2, 3, 4],
    "pass_at_k": 8,
    "rollout_k": 4,
    "rft_k": 4,
    "sft_epochs": 1,
    "g2_pass1_threshold": 0.30,
    "seeds": [0, 1, 2],
    # Dataset sizes are intentionally configurable for shakedowns vs. full runs.
    "n_forward_l1": 2500,
    "n_eval_per_cell": 100,
    "n_comp_train": 2500,
    "n_inverse_train": 2500,
    "n_inverse_l1to2_seen": 1200,
    "rollout_k": 4,
    "rft_k": 4,
    "sft_epochs": 1,
    "g2_pass1_threshold": 0.30,
}

HF_TOKEN = userdata.get("HF_TOKEN")
WANDB_API_KEY = userdata.get("WANDB_API_KEY")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("[skip] HF_TOKEN not found in Colab userdata; gated model downloads may fail")

STATE_PATH = ROOT / "state.json"

def state_load():
    if STATE_PATH.exists():
        with STATE_PATH.open("r", encoding="utf-8") as f:
            return json.load(f)
    return {}

state = state_load()

def state_save(key=None, value=True):
    global state
    state = state_load()
    if key is not None:
        state[key] = value
    tmp = STATE_PATH.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(state, f, indent=2, sort_keys=True)
    tmp.replace(STATE_PATH)
    return state

def exists(path):
    return Path(path).exists()

def done(key):
    return bool(state_load().get(key, False))

state_save()
print(f"ROOT={ROOT}")
print(f"state={STATE_PATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT=/content/drive/MyDrive/inverse-rl
state=/content/drive/MyDrive/inverse-rl/state.json


In [3]:
# Cell 2 — data generation (resumable JSONL outputs)
import subprocess
import sys
from pathlib import Path

DATASETS = {
    "fwd_l1_all.jsonl": ["--task", "forward", "--levels", "1", "--n", str(CFG["n_forward_l1"]), "--pool", "all", "--seed", str(CFG["seeds"][0])],
    "fwd_l1to4_eval.jsonl": ["--task", "forward", "--levels", "1,2,3,4", "--n", str(CFG["n_eval_per_cell"]), "--pool", "all", "--eval", "--seed", str(CFG["seeds"][0])],
    "fwd_l2_seen_comp_train.jsonl": ["--task", "forward", "--levels", "2", "--n", str(CFG["n_comp_train"]), "--pool", "seen", "--seed", str(CFG["seeds"][1])],
    "inv_l1_seen_train.jsonl": ["--task", "inverse", "--levels", "1", "--n", str(CFG["n_inverse_train"]), "--pool", "seen", "--seed", str(CFG["seeds"][1])],
    "inv_l1to4_eval.jsonl": ["--task", "inverse", "--levels", "1,2,3,4", "--n", str(CFG["n_eval_per_cell"]), "--pool", "all", "--eval", "--seed", str(CFG["seeds"][2])],
    "inv_l1to2_seen_optional.jsonl": ["--task", "inverse", "--levels", "1,2", "--n", str(CFG["n_inverse_l1to2_seen"]), "--pool", "seen", "--seed", str(CFG["seeds"][2])],
}

missing = [name for name in DATASETS if not (DATA_DIR / name).exists()]
if done("data") and not missing:
    print(f"[skip] data already present in {DATA_DIR}")
else:
    print(f"[run] generating {len(missing)} missing dataset(s) in {DATA_DIR}")
    for name in missing:
        out = DATA_DIR / name
        cmd = [sys.executable, "inverse_tasks.py", *DATASETS[name], "--out", str(out)]
        print("[run]", " ".join(cmd))
        subprocess.run(cmd, check=True)
    state_save("data", True)
    print("[run] data generation complete")


[skip] data already present in /content/drive/MyDrive/inverse-rl/data


In [4]:
# Cell 3 — evaluation utilities (HF/LoRA loading, vLLM generation, CSV outputs)
import gc
import json
import re
from pathlib import Path

import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams

from inverse_tasks import ALL_SKILLS
from skills_inverse import SKILLS
from verifier import forward_reward, inverse_reward

_ACTIVE_MODEL = None
_ACTIVE_LLM = None
_ACTIVE_LLM_KEY = None

def _safe_name(path_or_name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(path_or_name).rstrip("/"))[-120:]

def _read_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def _model_key(model):
    if isinstance(model, dict):
        return model["model_path"]
    return str(model)

def load_model(path_or_name, adapter=None):
    """Return a vLLM-ready model descriptor; merge an optional LoRA adapter once."""
    if adapter is None:
        model = {"model_path": str(path_or_name), "adapter": None, "name": _safe_name(path_or_name)}
        print(f"[skip] using base/merged model {model['model_path']}")
        return model

    merged_dir = CKPT_DIR / f"merged_{_safe_name(path_or_name)}__{_safe_name(adapter)}"
    if merged_dir.exists():
        print(f"[skip] merged adapter already exists at {merged_dir}")
        return {"model_path": str(merged_dir), "adapter": str(adapter), "name": _safe_name(merged_dir)}

    print(f"[run] merging LoRA adapter {adapter} into {path_or_name}")
    tokenizer = AutoTokenizer.from_pretrained(path_or_name, token=HF_TOKEN)
    base = AutoModelForCausalLM.from_pretrained(
        path_or_name,
        token=HF_TOKEN,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    merged = PeftModel.from_pretrained(base, adapter).merge_and_unload()
    merged.save_pretrained(merged_dir, safe_serialization=True)
    tokenizer.save_pretrained(merged_dir)
    del merged, base, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"model_path": str(merged_dir), "adapter": str(adapter), "name": _safe_name(merged_dir)}

def _get_llm(model):
    global _ACTIVE_LLM, _ACTIVE_LLM_KEY
    key = _model_key(model)
    if _ACTIVE_LLM is not None and _ACTIVE_LLM_KEY == key:
        return _ACTIVE_LLM
    if _ACTIVE_LLM is not None:
        del _ACTIVE_LLM
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f"[run] loading vLLM model {key}")
    _ACTIVE_LLM = LLM(
        model=key,
        tokenizer=key,
        trust_remote_code=True,
        dtype="bfloat16",
        max_model_len=CFG["max_prompt_len"] + CFG["max_completion_len"],
        gpu_memory_utilization=0.60,
        enforce_eager=True,
      )
    _ACTIVE_LLM_KEY = key
    return _ACTIVE_LLM

def generate(prompts, n=1, model=None):
    """Generate n completions per prompt with vLLM; call eval_task/load_model first or pass model."""
    global _ACTIVE_MODEL
    if model is not None:
        _ACTIVE_MODEL = model
    if _ACTIVE_MODEL is None:
        raise ValueError("No active model. Pass model=... or call eval_task(model, ...) first.")
    llm = _get_llm(_ACTIVE_MODEL)
    params = SamplingParams(
        n=n,
        temperature=0.7 if n > 1 else 0.0,
        top_p=0.95,
        max_tokens=CFG["max_completion_len"],
    )
    outputs = llm.generate(prompts, params)
    return [[choice.text for choice in out.outputs] for out in outputs]

def _cells_from_arg(cells):
    if isinstance(cells, (str, Path)):
        return _read_jsonl(cells)
    return list(cells)

def eval_task(model, cells, task, k=None, out_name=None):
    """Evaluate a task and save tidy metrics: level, split, pass@1, pass@k."""
    global _ACTIVE_MODEL
    k = int(k or CFG["pass_at_k"])
    problems = _cells_from_arg(cells)
    model_name = model.get("name", _safe_name(_model_key(model))) if isinstance(model, dict) else _safe_name(model)
    out_csv = RESULTS_DIR / (out_name or f"eval_{task}_{model_name}_k{k}.csv")
    if out_csv.exists():
        print(f"[skip] loading existing eval results from {out_csv}")
        return pd.read_csv(out_csv)

    print(f"[run] evaluating {task} on {len(problems)} problems with k={k}")
    _ACTIVE_MODEL = model
    reward_fn = forward_reward if task == "forward" else inverse_reward
    grouped = {}
    prompts = [p["prompt"] for p in problems]
    completions = generate(prompts, n=k)
    for problem, samples in zip(problems, completions):
        rewards = [reward_fn(sample, problem) for sample in samples]
        level = int(problem.get("level", 0))
        split = problem.get("eval_split") or ("seen" if problem.get("skills_seen") else "held_out")
        key = (level, split)
        grouped.setdefault(key, {"p1": [], "pk": []})
        grouped[key]["p1"].append(float(rewards[0] == 1.0) if rewards else 0.0)
        grouped[key]["pk"].append(float(any(r == 1.0 for r in rewards)))

    rows = []
    for (level, split), vals in sorted(grouped.items()):
        rows.append({
            "level": level,
            "split": split,
            "pass@1": sum(vals["p1"]) / len(vals["p1"]),
            f"pass@{k}": sum(vals["pk"]) / len(vals["pk"]),
            "n": len(vals["p1"]),
        })
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    state_save(out_csv.stem, True)
    print(f"[run] saved {out_csv}")
    return df

def forward_accuracy(model, eval_jsonl=None, k=1, out_name=None):
    """Forward pass@1 by skill and tier on level-1 forward eval examples."""
    eval_jsonl = eval_jsonl or (DATA_DIR / "fwd_l1to4_eval.jsonl")
    problems = [p for p in _read_jsonl(eval_jsonl) if int(p.get("level", 0)) == 1]
    model_name = model.get("name", _safe_name(_model_key(model))) if isinstance(model, dict) else _safe_name(model)
    out_csv = RESULTS_DIR / (out_name or f"forward_accuracy_{model_name}.csv")
    if out_csv.exists():
        print(f"[skip] loading existing forward accuracy from {out_csv}")
        return pd.read_csv(out_csv)

    print(f"[run] computing forward accuracy for {len(problems)} level-1 problems")
    global _ACTIVE_MODEL
    _ACTIVE_MODEL = model
    completions = generate([p["prompt"] for p in problems], n=k)
    rows = []
    for problem, samples in zip(problems, completions):
        skill = problem["chain"][0]
        tier = SKILLS[skill][4]
        rewards = [forward_reward(sample, problem) for sample in samples]
        rows.append({
            "skill": skill,
            "tier": tier,
            "correct": float(any(r == 1.0 for r in rewards)),
        })
    df = pd.DataFrame(rows).groupby(["skill", "tier"], as_index=False).agg(accuracy=("correct", "mean"), n=("correct", "size"))
    df.to_csv(out_csv, index=False)
    state_save(out_csv.stem, True)
    print(f"[run] saved {out_csv}")
    return df

print("[skip] eval utilities are defined; call load_model(), eval_task(), and forward_accuracy() as needed")


INFO 06-05 01:10:07 [__init__.py:216] Automatically detected platform cuda.
[skip] eval utilities are defined; call load_model(), eval_task(), and forward_accuracy() as needed


In [5]:
stage_0_smoke_test = False

if stage_0_smoke_test == True:

  model = load_model(CFG["shakedown_model"])

  tiny = _read_jsonl(DATA_DIR / "inv_l1to4_eval.jsonl")[:8]

  df = eval_task(
      model=model,
      cells=tiny,
      task="inverse",
      k=1,
      out_name="smoke_inv_1b_k1_tiny.csv",
  )

  df

In [6]:
# Cell 4 — Stage 1 forward RFT (EXECUTION_GUIDE Step 4)
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch
import wandb
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer

STAGE1_DIR = CKPT_DIR / "stage1_rft"
STAGE1_DATA_JSONL = RESULTS_DIR / "stage1_rft_data.jsonl"
STAGE1_STATS_CSV = RESULTS_DIR / "stage1_rft_stats.csv"
STAGE1_FORWARD_CSV = RESULTS_DIR / "stage1_forward.csv"


def _release_vllm():
    global _ACTIVE_LLM, _ACTIVE_LLM_KEY
    if _ACTIVE_LLM is not None:
        del _ACTIVE_LLM
        _ACTIVE_LLM = None
        _ACTIVE_LLM_KEY = None
        _ACTIVE_MODEL = None          # <-- add this line
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def _write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    tmp.replace(path)


def _stage1_stats(problems, kept_rows):
    attempted = defaultdict(int)
    kept = defaultdict(int)
    problem_kept = defaultdict(set)
    for idx, problem in enumerate(problems):
        skill = problem["chain"][0]
        tier = SKILLS[skill][4]
        key = (skill, tier)
        attempted[key] += 1
    for row in kept_rows:
        key = (row["skill"], row["tier"])
        kept[key] += 1
        problem_kept[key].add(row["problem_index"])
    stats_rows = []
    for skill, tier in sorted(attempted):
        key = (skill, tier)
        n_problems = attempted[key]
        n_kept = kept[key]
        n_any = len(problem_kept[key])
        stats_rows.append({
            "skill": skill,
            "tier": tier,
            "n_problems": n_problems,
            "n_kept": n_kept,
            "n_problems_with_any_kept": n_any,
            "keep_rate": n_kept / (n_problems * int(CFG["rollout_k"])),
            "coverage": n_any / n_problems,
        })
    return pd.DataFrame(stats_rows)


def _print_stage1_rft_summary(stats_df, n_kept):
    tier_summary = (
        stats_df.groupby("tier", as_index=False)
        .agg(
            n_problems=("n_problems", "sum"),
            n_kept=("n_kept", "sum"),
            n_problems_with_any_kept=("n_problems_with_any_kept", "sum"),
        )
        .sort_values("tier")
    )
    tier_summary["keep_rate"] = tier_summary["n_kept"] / (tier_summary["n_problems"] * int(CFG["rollout_k"]))
    tier_summary["coverage"] = tier_summary["n_problems_with_any_kept"] / tier_summary["n_problems"]
    print("Stage-1 RFT rejection-sampling yield by tier:")
    print(tier_summary[["tier", "keep_rate", "coverage", "n_kept", "n_problems"]].to_string(index=False))
    print(f"Stage-1 RFT total kept examples: {n_kept}")
    return tier_summary


def _log_stage1_rft_summary_to_wandb(tier_summary, n_kept):
    run = wandb.run
    if run is None:
        return
    run.config.update({
        "stage1_n_kept": int(n_kept),
        "stage1_rollout_k": int(CFG["rollout_k"]),
    }, allow_val_change=True)
    run.summary["stage1_n_kept"] = int(n_kept)
    for row in tier_summary.to_dict("records"):
        tier = row["tier"]
        run.config.update({
            f"stage1_tier_{tier}_keep_rate": float(row["keep_rate"]),
            f"stage1_tier_{tier}_coverage": float(row["coverage"]),
        }, allow_val_change=True)
        run.summary[f"stage1_tier_{tier}_keep_rate"] = float(row["keep_rate"])
        run.summary[f"stage1_tier_{tier}_coverage"] = float(row["coverage"])


def _build_stage1_rft_data():
    problems = _read_jsonl(DATA_DIR / "fwd_l1_all.jsonl")
    if STAGE1_DATA_JSONL.exists() and STAGE1_STATS_CSV.exists():
        print(f"[skip] loading existing Stage-1 RFT corpus from {STAGE1_DATA_JSONL}")
        kept_rows = _read_jsonl(STAGE1_DATA_JSONL)
        stats_df = pd.read_csv(STAGE1_STATS_CSV)
        tier_summary = _print_stage1_rft_summary(stats_df, len(kept_rows))
        return kept_rows, stats_df, tier_summary

    print(f"[run] Stage-1 rollout: {CFG['rollout_k']} samples/problem from base model using gen_prompt")
    base_model = load_model(CFG["model_name"])
    prompts = []
    prompt_to_problem_index = []
    for idx, problem in enumerate(problems):
        if not problem.get("gen_prompt"):
            raise ValueError("forward Stage-1 rollout requires gen_prompt on every problem")
        prompts.append(problem["gen_prompt"])
        prompt_to_problem_index.append(idx)

    completions = generate(prompts, n=int(CFG["rollout_k"]), model=base_model)
    kept_rows_with_index = []
    for prompt_idx, samples in enumerate(completions):
        problem_idx = prompt_to_problem_index[prompt_idx]
        problem = problems[problem_idx]
        skill = problem["chain"][0]
        tier = SKILLS[skill][4]
        for sample in samples:
            if forward_reward(sample, problem) == 1.0:
                kept_rows_with_index.append({
                    "problem_index": problem_idx,
                    "chain": problem["chain"],
                    "level": problem["level"],
                    "tier": tier,
                    "skill": skill,
                    "input": problem["input"],
                    "output": problem["output"],
                    "prompt": problem["prompt"],
                    "completion": sample,
                    "source_prompt": "gen_prompt",
                })

    stats_df = _stage1_stats(problems, kept_rows_with_index)
    audit_rows = [{k: v for k, v in row.items() if k != "problem_index"} for row in kept_rows_with_index]
    _write_jsonl(STAGE1_DATA_JSONL, audit_rows)
    stats_df.to_csv(STAGE1_STATS_CSV, index=False)
    print(f"[run] saved {STAGE1_DATA_JSONL}")
    print(f"[run] saved {STAGE1_STATS_CSV}")
    tier_summary = _print_stage1_rft_summary(stats_df, len(audit_rows))
    return audit_rows, stats_df, tier_summary


def _sft_training_args(output_dir):
    common = dict(
        output_dir=str(output_dir),
        num_train_epochs=float(CFG["sft_epochs"]),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=float(CFG["lr"]),
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=10,
        save_strategy="epoch",
        report_to=["wandb"],
        run_name="stage1-rft",
    )
    try:
        from trl import SFTConfig
        return SFTConfig(
            **common,
            max_length=int(CFG["max_prompt_len"]) + int(CFG["max_completion_len"]),
            dataset_text_field="text",
            packing=False,
        )
    except Exception:
        from transformers import TrainingArguments
        return TrainingArguments(**common)


def _make_sft_trainer(model, tokenizer, train_dataset, peft_config, args):
    kwargs = dict(model=model, args=args, train_dataset=train_dataset, peft_config=peft_config)
    try:
        return SFTTrainer(**kwargs, processing_class=tokenizer)
    except TypeError:
        try:
            return SFTTrainer(
                **kwargs,
                tokenizer=tokenizer,
                dataset_text_field="text",
                max_seq_length=int(CFG["max_prompt_len"]) + int(CFG["max_completion_len"]),
                packing=False,
            )
        except TypeError:
            return SFTTrainer(**kwargs, tokenizer=tokenizer)


def _train_stage1_rft(kept_rows, tier_summary):
    if not kept_rows:
        raise RuntimeError("Stage-1 RFT kept zero examples; inspect stage1_rft_stats.csv before training")

    _release_vllm()
    adapter_dir = CKPT_DIR / "stage1_rft_adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    print(f"[run] Stage-1 LoRA-SFT on {len(kept_rows)} hidden-def examples")

    train_dataset = Dataset.from_list([
        {"text": row["prompt"] + "\n" + row["completion"]}
        for row in kept_rows
    ])
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"], token=HF_TOKEN, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        CFG["model_name"],
        token=HF_TOKEN,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    model.config.use_cache = False
    peft_config = LoraConfig(
        r=int(CFG["lora_r"]),
        lora_alpha=int(CFG["lora_r"]) * 2,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    )
    wandb.init(project="inverse-rl", name="stage1-rft", config=dict(CFG), reinit=True)
    _log_stage1_rft_summary_to_wandb(tier_summary, len(kept_rows))
    try:
        args = _sft_training_args(adapter_dir)
        trainer = _make_sft_trainer(model, tokenizer, train_dataset, peft_config, args)
        trainer.train()
        trainer.save_model(str(adapter_dir))
        print(f"[run] merging Stage-1 adapter and saving merged model to {STAGE1_DIR}")
        merged = trainer.model.merge_and_unload()
        STAGE1_DIR.mkdir(parents=True, exist_ok=True)
        merged.save_pretrained(STAGE1_DIR, safe_serialization=True)
        tokenizer.save_pretrained(STAGE1_DIR)
    finally:
        wandb.finish()
        del model, tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    state_save("stage1_rft", True)


def _gate_g1(stage1_model):
    df = forward_accuracy(stage1_model, out_name="stage1_forward.csv")
    mean_acc = float(df["accuracy"].mean()) if len(df) else 0.0
    tier_means = df.groupby("tier")["accuracy"].mean().sort_index()
    bad_tiers = tier_means[tier_means < 0.75]
    bad_skills = df[df["accuracy"] < 0.75].sort_values(["tier", "accuracy", "skill"])
    passed = mean_acc >= 0.90 and bad_tiers.empty
    print(f"Gate G1: {'PASS' if passed else 'WARN'} — mean accuracy={mean_acc:.3f}; tier means={tier_means.to_dict()}")
    if not passed:
        if not bad_tiers.empty:
            print("Offending tiers:")
            print(bad_tiers.to_string())
        if not bad_skills.empty:
            print("Offending skills (accuracy < 0.75):")
            print(bad_skills[["skill", "tier", "accuracy", "n"]].to_string(index=False))
    state_save("stage1_forward", True)
    return df


if STAGE1_DIR.exists():
    print(f"[skip] Cell 4 Stage 1 forward RFT: merged model exists at {STAGE1_DIR}")
    stage1_model = load_model(STAGE1_DIR)
else:
    print("[run] Cell 4 Stage 1 forward RFT")
    kept_rows, stats_df, tier_summary = _build_stage1_rft_data()
    _train_stage1_rft(kept_rows, tier_summary)
    stage1_model = load_model(STAGE1_DIR)

# Gate G1 is idempotent through forward_accuracy(..., out_name="stage1_forward.csv").
_gate_g1(stage1_model)


[run] Cell 4 Stage 1 forward RFT
[skip] loading existing Stage-1 RFT corpus from /content/drive/MyDrive/inverse-rl/results/stage1_rft_data.jsonl
Stage-1 RFT rejection-sampling yield by tier:
 tier  keep_rate  coverage  n_kept  n_problems
    1   0.234729  0.386207     953        1015
    2   0.230809  0.406346     902         977
    3   0.003445  0.013780       7         508
Stage-1 RFT total kept examples: 1862
[run] Stage-1 LoRA-SFT on 1862 hidden-def examples


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: shengweiming25 (shengweiming25-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


...

INFO 06-05 01:25:47 [default_loader.py:268] Loading weights took 5.11 seconds
INFO 06-05 01:25:48 [model_runner.py:1083] Model loading took 6.0160 GiB and 5.210442 seconds
INFO 06-05 01:25:49 [worker.py:290] Memory profiling takes 0.82 seconds
INFO 06-05 01:25:49 [worker.py:290] the current vLLM instance can use total_gpu_memory (79.25GiB) x gpu_memory_utilization (0.60) = 47.55GiB
INFO 06-05 01:25:49 [worker.py:290] model weights take 6.02GiB; non_torch_memory takes 0.00GiB; PyTorch activation peak memory takes 1.18GiB; the rest of the memory reserved for KV Cache is 40.36GiB.
INFO 06-05 01:25:50 [executor_base.py:114] # cuda blocks: 23613, # CPU blocks: 2340
INFO 06-05 01:25:50 [executor_base.py:119] Maximum concurrency for 2560 tokens per request: 147.58x
INFO 06-05 01:25:53 [worker.py:467] Free memory on device (78.69/79.25 GiB) on startup. Desired GPU memory utilization is (0.6, 47.55 GiB). Actual usage is 6.02 GiB for weight, 1.18 GiB for peak activation, 0.0 GiB for non-torch me

Adding requests:   0%|          | 0/200 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/200 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[run] saved /content/drive/MyDrive/inverse-rl/results/stage1_forward.csv
Gate G1: WARN — mean accuracy=0.000; tier means={1: 0.0, 2: 0.0, 3: 0.0}
Offending tiers:
tier
1    0.0
2    0.0
3    0.0
Offending skills (accuracy < 0.75):
                skill  tier  accuracy  n
           add_prefix     1       0.0  6
               atbash     1       0.0 11
    complement_digits     1       0.0  8
 duplicate_every_char     1       0.0  8
       fancy_brackets     1       0.0  5
              reverse     1       0.0  1
          shift_chars     1       0.0  7
         shift_digits     1       0.0 12
            swap_case     1       0.0  8
             wrap_tag     1       0.0  7
           add_suffix     2       0.0  4
     insert_separator     2       0.0  8
           mirror_str     2       0.0 10
           repeat_str     2       0.0  3
        reverse_words     2       0.0  8
           rotate_str     2       0.0  5
         rotate_words     2       0.0  6
            succ_char     2    

,skill,tier,accuracy,n
0,add_prefix,1,0.0,6
1,add_suffix,2,0.0,4
2,atbash,1,0.0,11
3,complement_digits,1,0.0,8
4,deterministic_shuffle,3,0.0,7
5,duplicate_every_char,1,0.0,8
6,fancy_brackets,1,0.0,5
7,insert_separator,2,0.0,8
8,mirror_str,2,0.0,10
9,positional_shift,3,0.0,16


In [7]:
!pip install "peft==0.13.2"

In [8]:
# Cell 5 — Stage 1.5 inverse baseline (reversal-curse gate; EXECUTION_GUIDE Step 4)
import json
from pathlib import Path

import pandas as pd

STAGE1_DIR = CKPT_DIR / "stage1_rft"
STAGE1P5_CSV = RESULTS_DIR / "stage1p5_inverse.csv"
STAGE1P5_TIER_CSV = RESULTS_DIR / "stage1p5_inverse_tier_detail.csv"


def _load_stage1_or_fail():
    if not STAGE1_DIR.exists():
        raise FileNotFoundError(f"Stage-1 checkpoint not found: {STAGE1_DIR}; run Cell 4 first")
    return load_model(STAGE1_DIR)


def _level1_inverse_cells():
    return [p for p in _read_jsonl(DATA_DIR / "inv_l1to4_eval.jsonl") if int(p.get("level", 0)) == 1]


def _stage1p5_tier_detail(model, cells, k):
    if STAGE1P5_TIER_CSV.exists():
        print(f"[skip] loading existing Stage-1.5 tier detail from {STAGE1P5_TIER_CSV}")
        return pd.read_csv(STAGE1P5_TIER_CSV)

    print("[run] computing Stage-1.5 per-problem inverse rewards for Gate G2 using hidden-def prompts")
    completions = generate([p["prompt"] for p in cells], n=int(k), model=model)
    rows = []
    for problem, samples in zip(cells, completions):
        rewards = [inverse_reward(sample, problem) for sample in samples]
        skill = problem["chain"][0]
        tier = SKILLS[skill][4]
        rows.append({
            "skill": skill,
            "tier": tier,
            "split": problem.get("eval_split") or ("seen" if problem.get("skills_seen") else "held_out"),
            "level": int(problem.get("level", 0)),
            "pass@1": float(rewards[0] == 1.0) if rewards else 0.0,
            f"pass@{int(k)}": float(any(r == 1.0 for r in rewards)),
        })
    df = pd.DataFrame(rows)
    df.to_csv(STAGE1P5_TIER_CSV, index=False)
    state_save(STAGE1P5_TIER_CSV.stem, True)
    print(f"[run] saved {STAGE1P5_TIER_CSV}")
    return df


def _gate_g2(tier_df):
    hard = tier_df[tier_df["tier"].isin([2, 3, "T2", "T3"])]
    t23_pass1 = float(hard["pass@1"].mean()) if len(hard) else 0.0
    threshold = float(CFG["g2_pass1_threshold"])
    passed = t23_pass1 < threshold
    tier_summary = tier_df.groupby("tier")["pass@1"].mean().sort_index()
    print(f"Gate G2: {'PASS' if passed else 'WARN'} — T2–T3 level-1 inverse pass@1={t23_pass1:.3f} (threshold < {threshold:.3f}); tier means={tier_summary.to_dict()}")
    if not passed:
        print("WARN: reversal-curse contrast is weak; consider restricting the headline analysis to T2–T3.")
    state_save("stage1p5_inverse", True)
    return passed


stage1_model = _load_stage1_or_fail()
level1_cells = _level1_inverse_cells()

tier_detail_df = _stage1p5_tier_detail(stage1_model, level1_cells, int(CFG["pass_at_k"]))
k = int(CFG["pass_at_k"])
stage1p5_df = (
    tier_detail_df.groupby("split", as_index=False)
    .agg(**{"pass@1": ("pass@1", "mean"), f"pass@{k}": (f"pass@{k}", "mean"), "n": ("pass@1", "size")})
)
stage1p5_df.insert(0, "level", 1)
stage1p5_df.to_csv(STAGE1P5_CSV, index=False)
print(stage1p5_df.to_string(index=False))

_gate_g2(tier_detail_df)


[skip] using base/merged model /content/drive/MyDrive/inverse-rl/ckpts/stage1_rft
[run] computing Stage-1.5 per-problem inverse rewards for Gate G2 using hidden-def prompts


Adding requests:   0%|          | 0/200 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1600 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

[run] saved /content/drive/MyDrive/inverse-rl/results/stage1p5_inverse_tier_detail.csv
 level    split  pass@1  pass@8   n
     1 held_out    0.00    0.01 100
     1     seen    0.03    0.09 100
Gate G2: PASS — T2–T3 level-1 inverse pass@1=0.000 (threshold < 0.300); tier means={1: 0.04225352112676056, 2: 0.0, 3: 0.0}


True

In [9]:
# Cell 6 — Stage 2 composition control / shared trainer (stub for EXECUTION_GUIDE Step 5)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 6 stub: Stage 2 composition control / shared trainer (stub for EXECUTION_GUIDE Step 5)")


[skip] Cell 6 stub: Stage 2 composition control / shared trainer (stub for EXECUTION_GUIDE Step 5)


In [10]:
# Cell 7 — Stage 2 inversion RL vs RFT (stub for EXECUTION_GUIDE Step 6)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 7 stub: Stage 2 inversion RL vs RFT (stub for EXECUTION_GUIDE Step 6)")


[skip] Cell 7 stub: Stage 2 inversion RL vs RFT (stub for EXECUTION_GUIDE Step 6)


In [11]:
# Cell 8 — Analysis, plots, and final exports (stub for EXECUTION_GUIDE Step 7)
# Intentionally empty for now. Implement this cell only when the corresponding [CODEX] step is requested.
print("[skip] Cell 8 stub: Analysis, plots, and final exports (stub for EXECUTION_GUIDE Step 7)")


[skip] Cell 8 stub: Analysis, plots, and final exports (stub for EXECUTION_GUIDE Step 7)
